# 🚦 AI Traffic Monitoring — Wrong-Way Vehicle Detection (YOLOv8 + ByteTrack)

**By Youssef Lasheen — Lasheen Academy**

This notebook detects vehicles in a video, tracks each one across frames, works out
which direction it is actually heading, and flags any vehicle moving **against** the
allowed direction of its lane as **"Wrong Way"** — with a live per-lane stats panel,
similar in spirit to professional traffic-analytics dashboards.

**Pipeline:**
1. **Detect** vehicles every frame with a pretrained YOLOv8 model (car / motorcycle / bus / truck).
2. **Track** each vehicle across frames with ByteTrack → every vehicle keeps one stable ID.
3. **Classify direction**: compare each vehicle's recent motion vector to the *allowed*
   direction of the lane/zone it's in.
4. **Flag wrong-way vehicles** in red, draw correctly-flowing ones in blue, and keep a
   running per-lane tally + a live "Clear / Wrong-Way!" status banner.

Runs end-to-end in Google Colab (free GPU recommended: `Runtime → Change runtime type → GPU`)
or on your own machine / Jupyter server.


## 0. Install dependencies

Skip this cell if you already have `ultralytics` and `opencv` installed locally.

In [1]:
!pip install -q ultralytics opencv-python-headless
print("Done.")

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

Done.


## 1. Get a video to analyze

You have two options:

- **Option A (default below):** download the same demo clip used to build this notebook —
  a real "wrong-way driveway entry" CCTV clip.
- **Option B:** use your own footage. In Colab, run the commented `files.upload()` line
  in the next cell and pick your `.mp4`; then set `SOURCE_VIDEO` to its filename.
  For a video that looks like a bird's-eye two-lane highway (the classic "Left Lane /
  Right Lane" style), see the **Highway template** near the end of this notebook.


In [2]:
import os, urllib.request

SOURCE_VIDEO = "wrongway.mp4"
DEMO_URL = "https://raw.githubusercontent.com/Ikshan-Tango/Vehicle_Wrong_Way_Detection_YoloV8/main/wrongway.mp4"

if not os.path.exists(SOURCE_VIDEO):
    print("Downloading demo clip...")
    urllib.request.urlretrieve(DEMO_URL, SOURCE_VIDEO)
print("Using source video:", SOURCE_VIDEO)

# --- Option B: uncomment to upload your own video in Google Colab ---
# from google.colab import files
# uploaded = files.upload()
# SOURCE_VIDEO = list(uploaded.keys())[0]


Using source video: wrongway.mp4


## 2. The detection engine

This is the reusable core: a `Zone` (a lane / monitored region with an allowed direction) and a `WrongWayDetector` that runs YOLO + tracking and draws everything.

In [3]:
from __future__ import annotations
import time
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import Deque, Dict, List, Optional, Sequence, Tuple, Callable

import cv2
import numpy as np
from ultralytics import YOLO

Point = Tuple[float, float]
VEHICLE_CLASSES = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}

_DIRECTION_VECTORS = {"up": (0.0, -1.0), "down": (0.0, 1.0), "left": (-1.0, 0.0), "right": (1.0, 0.0)}

def _resolve_direction(direction) -> np.ndarray:
    vec = _DIRECTION_VECTORS[direction.lower()] if isinstance(direction, str) else direction
    v = np.array(vec, dtype=np.float32)
    n = np.linalg.norm(v)
    return v / n if n > 1e-6 else v

In [4]:
@dataclass
class Zone:
    '''A monitored lane/region. polygon=None means the whole frame.
    allowed_direction: up/down/left/right or an (dx, dy) tuple.'''
    label: str
    allowed_direction: object
    polygon: Optional[List[Point]] = None
    panel_anchor: Point = (20, 20)
    box_color: Tuple[int, int, int] = (255, 176, 0)

    def __post_init__(self):
        self._dir_vec = _resolve_direction(self.allowed_direction)
        self._poly_np = np.array(self.polygon, dtype=np.int32) if self.polygon else None

    def contains(self, pt: Point) -> bool:
        if self._poly_np is None:
            return True
        return cv2.pointPolygonTest(self._poly_np, pt, False) >= 0

    def heading_is_correct(self, motion_vec: np.ndarray, cos_threshold: float) -> bool:
        norm = np.linalg.norm(motion_vec)
        if norm < 1e-6:
            return True
        cos_sim = float(np.dot(motion_vec / norm, self._dir_vec))
        return cos_sim >= cos_threshold

In [5]:
@dataclass
class _TrackState:
    history: Deque[Point] = field(default_factory=lambda: deque(maxlen=18))
    cls_name: str = "car"
    zone_label: Optional[str] = None
    status: str = "unclassified"
    stopped: bool = False
    last_seen_frame: int = 0
    counted: bool = False


class WrongWayDetector:
    def __init__(self, zones, model_path="yolov8n.pt", process_size=(1280, 720),
                 conf=0.35, tracker="bytetrack.yaml", classes=tuple(VEHICLE_CLASSES.keys()),
                 min_displacement_px=14.0, cos_threshold=-0.10, stop_speed_px=1.5):
        self.zones = list(zones)
        self.model = YOLO(model_path)
        self.process_size = process_size
        self.conf = conf
        self.tracker = tracker
        self.classes = list(classes)
        self.min_displacement_px = min_displacement_px
        self.cos_threshold = cos_threshold
        self.stop_speed_px = stop_speed_px
        self.tracks: Dict[int, _TrackState] = defaultdict(_TrackState)
        self.zone_correct_ids = {z.label: set() for z in self.zones}
        self.zone_wrong_ids = {z.label: set() for z in self.zones}
        self.frame_idx = 0

    def _find_zone(self, pt):
        for z in self.zones:
            if z.contains(pt):
                return z
        return None

    def _update_track(self, track_id, box, cls_name):
        x1, y1, x2, y2 = box
        cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
        st = self.tracks[track_id]
        st.history.append((cx, cy))
        st.cls_name = cls_name
        st.last_seen_frame = self.frame_idx
        zone = self._find_zone((cx, cy))
        if zone is None:
            return st, None
        st.zone_label = zone.label
        if len(st.history) >= 2:
            p_old = np.array(st.history[0]); p_new = np.array(st.history[-1])
            motion_vec = p_new - p_old
            total_disp = float(np.linalg.norm(motion_vec))
            p_prev = np.array(st.history[-2])
            recent_speed = float(np.linalg.norm(p_new - p_prev))
            st.stopped = recent_speed < self.stop_speed_px
            if total_disp >= self.min_displacement_px:
                is_correct = zone.heading_is_correct(motion_vec, self.cos_threshold)
                st.status = "correct" if is_correct else "wrong"
                if not st.counted:
                    st.counted = True
                    (self.zone_correct_ids if is_correct else self.zone_wrong_ids)[zone.label].add(track_id)
                elif not is_correct and track_id not in self.zone_wrong_ids[zone.label]:
                    self.zone_wrong_ids[zone.label].add(track_id)
                    self.zone_correct_ids[zone.label].discard(track_id)
                    st.status = "wrong"
        return st, zone

    def process_frame(self, frame):
        self.frame_idx += 1
        if self.process_size is not None:
            frame = cv2.resize(frame, self.process_size)
        results = self.model.track(frame, persist=True, conf=self.conf, classes=self.classes,
                                    tracker=self.tracker, verbose=False)[0]
        annotated = frame.copy()
        self._draw_zone_outlines(annotated)
        active_wrong = defaultdict(int)
        if results.boxes is not None and results.boxes.id is not None:
            boxes = results.boxes.xyxy.cpu().numpy()
            ids = results.boxes.id.cpu().numpy().astype(int)
            clss = results.boxes.cls.cpu().numpy().astype(int)
            for box, tid, c in zip(boxes, ids, clss):
                cls_name = VEHICLE_CLASSES.get(c, "vehicle")
                st, zone = self._update_track(tid, box, cls_name)
                self._draw_vehicle(annotated, box, tid, st, zone)
                if zone is not None and st.status == "wrong":
                    active_wrong[zone.label] += 1
        self._draw_stats_panels(annotated, active_wrong)
        return annotated

    def _draw_zone_outlines(self, frame):
        for z in self.zones:
            if z.polygon:
                pts = np.array(z.polygon, dtype=np.int32).reshape(-1, 1, 2)
                cv2.polylines(frame, [pts], True, z.box_color, 2, cv2.LINE_AA)

    def _draw_vehicle(self, frame, box, tid, st, zone):
        x1, y1, x2, y2 = map(int, box)
        color = (60, 40, 230) if st.status == "wrong" else (255, 176, 0) if st.status == "correct" else (0, 200, 255)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        cv2.putText(frame, f"{st.cls_name} ID:{tid}", (x1, max(20, y1 - 22)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)
        if st.status == "wrong":
            tag = "Wrong Way (Stopped)" if st.stopped else "Wrong Way"
            cv2.putText(frame, tag, (x1, max(40, y1 - 4)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (60, 40, 230), 2, cv2.LINE_AA)

    def _draw_stats_panels(self, frame, active_wrong):
        for z in self.zones:
            correct_n = len(self.zone_correct_ids[z.label]); wrong_n = len(self.zone_wrong_ids[z.label])
            live_wrong = active_wrong.get(z.label, 0)
            x, y = int(z.panel_anchor[0]), int(z.panel_anchor[1]); w, h = 260, 100
            overlay = frame.copy()
            cv2.rectangle(overlay, (x, y), (x + w, y + h), (20, 20, 20), -1)
            frame[:] = cv2.addWeighted(overlay, 0.55, frame, 0.45, 0)
            cv2.putText(frame, z.label, (x + 12, y + 24), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.putText(frame, f"Correct: {correct_n}", (x + 12, y + 50), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 176, 0), 2, cv2.LINE_AA)
            cv2.putText(frame, f"Wrong Way: {wrong_n}", (x + 12, y + 72), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (60, 40, 230), 2, cv2.LINE_AA)
            status_txt, status_col = (f"Status: {live_wrong} Wrong-Way!", (60, 40, 230)) if live_wrong > 0 else ("Status: Clear", (0, 200, 0))
            cv2.putText(frame, status_txt, (x + 12, y + 94), cv2.FONT_HERSHEY_SIMPLEX, 0.55, status_col, 2, cv2.LINE_AA)

    def summary(self):
        return {z.label: {"correct": len(self.zone_correct_ids[z.label]), "wrong_way": len(self.zone_wrong_ids[z.label])} for z in self.zones}


def run_on_video(detector, source_path, output_path, fps_override=None, progress_cb=None, max_frames=None):
    cap = cv2.VideoCapture(source_path)
    if not cap.isOpened():
        raise IOError(f"Could not open source video: {source_path}")
    src_fps = fps_override or cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_frames:
        total = min(total, max_frames)
    w, h = detector.process_size or (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), src_fps, (w, h))
    t0 = time.time(); i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        writer.write(detector.process_frame(frame))
        i += 1
        if progress_cb:
            progress_cb(i, total)
        if max_frames and i >= max_frames:
            break
    cap.release(); writer.release()
    print(f"Processed {i} frames in {time.time()-t0:.1f}s -> {output_path}")
    return detector.summary()

print("Engine loaded ✅")

Engine loaded ✅


## 3. Configure the zones for THIS demo video

The demo clip is a home CCTV camera watching a one-way driveway entrance. We treat the
whole frame as one monitored zone: normal traffic moves **away** from the camera
(up-frame); a vehicle heading back **toward** the camera / private driveway is flagged.


In [6]:
DRIVEWAY_ZONES = [
    Zone(label="Driveway Entrance", allowed_direction="up", polygon=None,
         panel_anchor=(20, 20), box_color=(255, 176, 0)),
]
print("Zones ready:", [z.label for z in DRIVEWAY_ZONES])

Zones ready: ['Driveway Entrance']


## 4. Run detection on the video

On Colab with a GPU this takes well under a minute for a short clip. On CPU it's a few FPS — for a quick test, set `max_frames` to something small (e.g. `150`) first, then re-run with `max_frames=None` for the full video.

In [7]:
detector = WrongWayDetector(zones=DRIVEWAY_ZONES, model_path="yolov8n.pt", process_size=(1280, 720))

def progress(i, total):
    if i % 50 == 0 or i == total:
        print(f"{i}/{total} frames")

summary = run_on_video(detector, SOURCE_VIDEO, "traffic_analysis_raw.mp4",
                        progress_cb=progress, max_frames=None)
print("\nFinal summary:", summary)

50/525 frames


100/525 frames


150/525 frames


200/525 frames


250/525 frames


...

350/525 frames


400/525 frames


450/525 frames


500/525 frames


525/525 frames
Processed 525 frames in 52.2s -> traffic_analysis_raw.mp4

Final summary: {'Driveway Entrance': {'correct': 3, 'wrong_way': 4}}


## 5. Re-encode for playback & preview the result

OpenCV's writer isn't always browser-friendly, so we re-encode with `ffmpeg` (H.264) — this is also the exact file you'd screen-record or upload for a LinkedIn post.

In [8]:
import subprocess
subprocess.run([
    "ffmpeg", "-y", "-i", "traffic_analysis_raw.mp4",
    "-c:v", "libx264", "-pix_fmt", "yuv420p", "-preset", "medium", "-crf", "21",
    "-movflags", "+faststart", "traffic_analysis_output.mp4"
], check=True)

from IPython.display import Video
Video("traffic_analysis_output.mp4", embed=True, width=800)

ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

[mp4 @ 0x5577ac2266c0] Starting second pass: moving the moov atom to the beginning of the file
[out#0/mp4 @ 0x5577ac225840] video:3846kB audio:0kB subtitle:0kB other streams:0kB global headers:0kB muxing overhead: 0.180539%
frame=  525 fps= 47 q=27.0 Lsize=    3853kB time=00:00:17.41 bitrate=1812.4kbits/s speed=1.56x    
[libx264 @ 0x5577ac227340] frame I:3     Avg QP:16.99  size:209589
[libx264 @ 0x5577ac227340] frame P:156   Avg QP:19.35  size: 19255
[libx264 @ 0x5577ac227340] frame B:366   Avg QP:21.48  size:   835
[libx264 @ 0x5577ac227340] consecutive B-frames:  3.8%  8.0%  5.1% 83.0%
[libx264 @ 0x5577ac227340] mb I  I16..4:  5.0% 87.4%  7.5%
[libx264 @ 0x5577ac227340] mb P  I16..4:  0.2%  1.5%  0.3%  P16..4: 24.0%  3.7%  4.9%  0.0%  0.0%    skip:65.4%
[libx264 @ 0x5577ac227340] mb B  I16..4:  0.0%  0.1%  0.0%  B16..8:  7.4%  0.4%  0.1%  direct: 0.1%  skip:91.8%  L0:43.5% L1:53.6% BI: 2.9%
[libx264 @ 0x5577ac227340] 8x8 transform intra:78.9% inter:87.3%
[libx264 @ 0x5577ac227340] 

## 6. (Optional) Highway two-lane template

To reproduce the classic bird's-eye **"Left Lane Stats / Right Lane Stats"** divided-highway
look — two lanes with opposite-direction traffic, split down the middle — point this at your
own drone/CCTV highway footage and tune the polygon + split position to match your camera.

```python
PROCESS_W, PROCESS_H = 1280, 720
split_x = PROCESS_W // 2

HIGHWAY_ZONES = [
    Zone(label="Left Lane", allowed_direction="down",
         polygon=[(0, 0), (split_x, 0), (split_x, PROCESS_H), (0, PROCESS_H)],
         panel_anchor=(20, 20), box_color=(255, 176, 0)),
    Zone(label="Right Lane", allowed_direction="up",
         polygon=[(split_x, 0), (PROCESS_W, 0), (PROCESS_W, PROCESS_H), (split_x, PROCESS_H)],
         panel_anchor=(PROCESS_W - 280, 20), box_color=(0, 200, 255)),
]

highway_detector = WrongWayDetector(zones=HIGHWAY_ZONES, model_path="yolov8n.pt")
run_on_video(highway_detector, "your_highway_video.mp4", "highway_output.mp4")
```

**Tuning tips**
- `min_displacement_px` — raise it if slow-moving/parked vehicles get misclassified from tiny jitter.
- `cos_threshold` — closer to `-1.0` = more lenient (only near-total reversals count as wrong-way); closer to `0` = stricter (anything past ~90° off is flagged).
- `conf` — YOLO confidence threshold; raise it if you get false-positive boxes on background clutter.
- Swap `yolov8n.pt` → `yolov8s.pt` / `yolov8m.pt` for higher accuracy at the cost of speed (use a GPU runtime for the bigger models).


## 7. Recording a demo video for LinkedIn

1. Run this notebook end-to-end (Colab with GPU = fastest) and download `traffic_analysis_output.mp4`.
2. Either post that file directly, **or** screen-record yourself narrating over it playing
   (OBS Studio / QuickTime / Xbox Game Bar all work) for a more personal "here's how it works" post.
3. Good caption angles (matching the tone of posts like this): what the project *does*,
   what you *learned* moving from plain detection to behavior-aware analytics, and a
   question inviting engagement — e.g. *"What additional signal would you want a
   traffic-monitoring AI to surface?"*
4. Suggested hashtags: `#ComputerVision #YOLO #TrafficAnalytics #SmartCities #DeepLearning #Python`
